# M4-L4 — Packaging-signal-mediated co-assembly (stochastic)

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in `M4_theory_compendium.pdf`, a separate document that is not included in this repository.

**What this layer does.** L1–L3 assemble empty shells; L4 assembles *around cargo*. A 12-face dodecahedral lattice stands in for the T=3 shell, and each face carries a packaging signal with its own affinity. Two reaction orders, RNA-mediated recruitment and protein–protein addition, are both treated at detailed balance, and the system is simulated with the Gillespie algorithm.

**Result relevant to BruSafe.** The optimum in pac-site affinity reverses between a 12-signal viral genome and the sparse 3-pac-site cargo used here: weak signals are favoured for a dense cargo and strong signals for a sparse one. Affinity can be changed in the laboratory, so it is a design variable.

**Runtime:** ~1–3 min for the affinity scans.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile M4_ps_coassembly.py
"""
Packaging-signal-mediated co-assembly (Dykeman, Stockley & Twarock,
Phys. Rev. E 87, 022717, 2013; doi:10.1103/physreve.87.022717)
re-implemented for BruSafe / CAPSID Module 4, iGEM 2026.

Dodecahedral study system: 12 pentamer sites on the shell, 12 packaging signals
along the cargo. Coat protein is recruited to the cargo by second-order PS binding,
then incorporated into the shell by first-order protein-protein association.
Off-rates follow from detailed balance on both reaction orders. The cargo threads
the shell as a self-avoiding walk on the face-adjacency graph, so a growing end can
dead-end -- that is the kinetic trap, and escape requires breaking protein bonds.

Validated against two exact combinatorial invariants of the published model:
  73 unique protein-only intermediates and 1264 distinct Hamiltonian paths.

Run:  python M4_ps_coassembly.py
Deps: numpy + stdlib (math, random, itertools).
"""
import math, random
import numpy as np
from itertools import combinations

RT = 0.593          # kcal/mol at 298 K
N_REF = 3000        # paper's reference ensemble size (concentration anchor)
POS = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]


def dodecahedral_adjacency():
    """Face adjacency of the dodecahedron = vertex adjacency of the icosahedron."""
    phi = (1 + 5**0.5)/2
    V = []
    for s1 in (1, -1):
        for s2 in (1, -1):
            V += [(0, s1*1, s2*phi), (s1*1, s2*phi, 0), (s2*phi, 0, s1*1)]
    V = np.array(sorted(set(V)))
    D = np.linalg.norm(V[:, None, :] - V[None, :, :], axis=-1)
    dmin = np.min(D[D > 1e-9])
    adj = [tuple(np.where(np.abs(D[i] - dmin) < 1e-9)[0]) for i in range(12)]
    return V, adj


V, ADJ = dodecahedral_adjacency()


class Fenwick:
    """Binary indexed tree over per-cargo propensities: O(log N) sample and update."""
    __slots__ = ("n", "t")
    def __init__(self, n): self.n = n; self.t = [0.0]*(n+1)
    def add(self, i, d):
        i += 1
        while i <= self.n: self.t[i] += d; i += i & -i
    def total(self): return self.t[self.n] if self.n else 0.0
    def find(self, x):
        i = 0; bit = 1 << (self.n.bit_length())
        while bit:
            if i + bit <= self.n and self.t[i+bit] <= x: i += bit; x -= self.t[i]
            bit >>= 1
        return i


class PSAssembly:
    """PS-mediated co-assembly on the dodecahedral study system.

    Trap = the threading walk dead-ends (growing end has no unvisited neighbour);
    escape only by breaking protein-protein bonds at rate k2P.
    """
    POS = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]
    def __init__(self, dG_R, dG_P=-2.5, k1P=100.0, k1R=0.0024, C=1e-6, n_rna=200,
                 seed=0, nuc_pos=-4, adj=None):
        self.rng = random.Random(seed); self.adj = adj if adj is not None else ADJ
        self.N = n_rna
        b = 1.0/RT; self.b = b; self.k1R = k1R; self.k1P = k1P
        self.k2R = [k1R/(C*math.exp(-b*g)) for g in dG_R]     # detailed balance, 2nd order
        self.k2P = k1P*math.exp(b*dG_P)                        # detailed balance, 1st order
        self.sn = self.POS.index(nuc_pos)
        self.nP = n_rna*12; self.t = 0.0; self.ncap = 0
        self.occ  = [[False]*12 for _ in range(n_rna)]         # PS site occupied by CP
        self.path = [[] for _ in range(n_rna)]                 # [(ps_index, face)]
        self.vis  = [set() for _ in range(n_rna)]              # faces already threaded
        self.done = [False]*n_rna
        self.hp = []                                           # completed Hamiltonian paths
        self.a = [0.0]*n_rna; self.T = Fenwick(n_rna)
        for i in range(n_rna): self._sync(i)

    def _ev(self, i):
        """Enumerate (rate, event) pairs available to cargo i."""
        if self.done[i]: return []
        occ, p, vis = self.occ[i], self.path[i], self.vis[i]; e = []
        nfree = 12 - sum(occ)
        if nfree and self.nP > 0: e.append((self.k1R*self.nP*nfree, ("bind", 0)))
        inp = {s for s, _ in p}
        for s in range(12):
            if occ[s] and s not in inp: e.append((self.k2R[s], ("unb", s)))
        if p:
            (sa, fa), (sb, fb) = p[0], p[-1]
            if sa > 0 and occ[sa-1] and any(g not in vis for g in self.adj[fa]):
                e.append((self.k1P, ("gL", 0)))
            if sb < 11 and occ[sb+1] and any(g not in vis for g in self.adj[fb]):
                e.append((self.k1P, ("gR", 0)))
            e += [(self.k2P, ("bL", 0)), (self.k2P, ("bR", 0))]
        elif occ[self.sn]:
            if self.sn > 0 and occ[self.sn-1]: e.append((self.k1P, ("nuc", self.sn-1)))
            if self.sn < 11 and occ[self.sn+1]: e.append((self.k1P, ("nuc", self.sn)))
        return e

    def _sync(self, i):
        tot = sum(w for w, _ in self._ev(i)); self.T.add(i, tot - self.a[i]); self.a[i] = tot

    def _apply(self, i, ev):
        k, arg = ev; occ, p, vis = self.occ[i], self.path[i], self.vis[i]
        if k == "bind":
            s = self.rng.choice([q for q in range(12) if not occ[q]]); occ[s] = True; self.nP -= 1
        elif k == "unb":
            occ[arg] = False; self.nP += 1
        elif k == "nuc":
            f0 = self.rng.randrange(12); f1 = self.rng.choice(self.adj[f0])
            self.path[i] = [(arg, f0), (arg+1, f1)]; self.vis[i] = {f0, f1}
        elif k in ("gL", "gR"):
            s, f = (p[0][0]-1, p[0][1]) if k == "gL" else (p[-1][0]+1, p[-1][1])
            c = [g for g in self.adj[f] if g not in vis]
            if not c: return                                   # dead end = kinetic trap
            g = self.rng.choice(c); vis.add(g)
            (p.insert(0, (s, g)) if k == "gL" else p.append((s, g)))
            if len(p) == 12:
                self.done[i] = True; self.ncap += 1; self.hp.append(tuple(f for _, f in p))
        else:
            if not p: return
            s, f = (p.pop(0) if k == "bL" else p.pop()); vis.discard(f)
            if len(p) == 1:
                s2, f2 = p.pop(); vis.discard(f2)

    def step(self):
        A = self.T.total()
        if A <= 0: return False
        self.t += -math.log(self.rng.random())/A               # Gillespie time increment
        i = self.T.find(self.rng.random()*A)
        ev = self._ev(i); tot = sum(w for w, _ in ev)
        if tot <= 0: self._sync(i); return True
        x = self.rng.random()*tot; c = 0.0
        for w, e in ev:
            c += w
            if x < c: break
        self._apply(i, e); self._sync(i); return True

    def run(self, t_end, max_events=4_000_000):
        n = 0
        while self.t < t_end and n < max_events:
            if not self.step(): break
            n += 1
        return dict(t=self.t, events=n, y=self.ncap/self.N, capped=n >= max_events)


def rescale_concentration(sim, n_rna):
    """Hold the RNA-CP on-rate at the paper's concentration regardless of ensemble size."""
    sim.k1R = sim.k1R*(N_REF/n_rna)
    for i in range(n_rna): sim._sync(i)
    return sim


def run(dG_R, nuc_pos=-4, n_rna=250, seed=0, dG_P=-2.5, k1P=100.0, C=1e-6, t_end=300.0):
    """One stochastic co-assembly run; returns fractional capsid yield."""
    s = PSAssembly(dG_R, dG_P=dG_P, k1P=k1P, C=C, n_rna=n_rna, seed=seed, nuc_pos=nuc_pos)
    rescale_concentration(s, n_rna)
    return s.run(t_end)["y"]


# ------------------------------------------------------------------ validation
def count_hamiltonian_paths(adj=ADJ, group_order=60):
    """Distinct threading paths modulo the icosahedral rotation group."""
    n = [0]
    def dfs(v, visited, k):
        if k == 12:
            n[0] += 1; return
        for w in adj[v]:
            if not (visited >> w) & 1:
                dfs(w, visited | (1 << w), k+1)
    for s in range(12):
        dfs(s, 1 << s, 1)
    return n[0]//group_order                         # 75840 directed / 60


def rotation_group(adj=ADJ, V=V):
    """Icosahedral rotation group as permutations of the 12 faces."""
    Vn = V/np.linalg.norm(V, axis=1, keepdims=True)
    B0 = np.array([Vn[0], Vn[adj[0][0]], np.cross(Vn[0], Vn[adj[0][0]])]).T
    perms = set()
    for i in range(12):
        for j in adj[i]:
            Bi = np.array([Vn[i], Vn[j], np.cross(Vn[i], Vn[j])]).T
            R = Bi @ np.linalg.inv(B0)
            img = Vn @ R.T
            p = tuple(int(np.argmax(img[k] @ Vn.T)) for k in range(12))
            if sorted(p) == list(range(12)):
                perms.add(p)
    return perms


def count_intermediates(adj=ADJ, V=V):
    """Connected face subsets up to the rotation group, by number of pentamers."""
    perms = rotation_group(adj, V)
    def canon(mask):
        best = mask
        for p in perms:
            m = 0
            for f in range(12):
                if (mask >> f) & 1: m |= 1 << p[f]
            best = min(best, m)
        return best
    def connected(mask):
        fs = [f for f in range(12) if (mask >> f) & 1]
        if not fs: return False
        seen, st = {fs[0]}, [fs[0]]
        while st:
            v = st.pop()
            for w in adj[v]:
                if (mask >> w) & 1 and w not in seen:
                    seen.add(w); st.append(w)
        return len(seen) == len(fs)
    per_n = {}
    for k in range(1, 13):
        cs = set()
        for comb in combinations(range(12), k):
            m = 0
            for f in comb: m |= 1 << f
            if connected(m): cs.add(canon(m))
        per_n[k] = len(cs)
    return per_n, len(perms)


if __name__ == "__main__":
    per_n, gorder = count_intermediates()
    print(f"rotation group order   {gorder:>6}   (published 60)")
    print(f"unique intermediates   {sum(per_n.values()):>6}   (published 73)")
    print(f"Hamiltonian paths      {count_hamiltonian_paths():>6}   (published 1264)")

    print("\nuniform PS affinity, 12 sites (viral-genome-like cargo):")
    for g in (-1.0, -2.0, -3.0, -6.0, -12.0):
        y = np.mean([run([g]*12, nuc_pos=1, seed=s) for s in (1, 2, 3)])
        print(f"   dG_PS={g:+6.1f}   yield {y*100:5.1f}%")

    print("\nBruSafe 3-pac-site cargo (other sites non-binding):")
    idx = [0, 6, 11]
    for g in (-2.0, -6.0, -12.0):
        dG = [-0.05]*12
        for i in idx: dG[i] = g
        y = np.mean([run(dG, nuc_pos=POS[idx[1]], seed=s) for s in (1, 2, 3)])
        print(f"   dG_PS={g:+6.1f}   yield {y*100:5.1f}%")


## 1. Exact combinatorial invariants — the engine validating itself

Before trusting any yield number, the lattice is checked against three published integers: the icosahedral rotation group order, the number of symmetry-distinct intermediates, and the number of Hamiltonian threading paths.

In [ ]:
import M4_ps_coassembly as L4

per_n, gorder = L4.count_intermediates()
print(f'rotation group order   {gorder:>6}   (published 60)')
print(f'unique intermediates   {sum(per_n.values()):>6}   (published 73)')
print(f'Hamiltonian paths      {L4.count_hamiltonian_paths():>6}   (published 1264)')

## 2. Uniform affinity — a viral-genome-like cargo with 12 signals

In [ ]:
gs = [-1.0, -2.0, -3.0, -6.0, -12.0]
y12 = []
for g in gs:
    y = np.mean([L4.run([g]*12, nuc_pos=1, seed=s) for s in (1, 2, 3)])
    y12.append(y); print(f'dG_PS = {g:+6.1f} kcal/mol   yield {y*100:5.1f}%')

## 3. The BruSafe cargo — only 3 pac-sites

Non-signal faces get a near-zero affinity, so recruitment has to come from the three engineered sites plus protein-protein addition.

In [ ]:
idx = [0, 6, 11]
gs3 = [-2.0, -6.0, -9.0, -12.0]
y3 = []
for g in gs3:
    dG = [-0.05]*12
    for i in idx: dG[i] = g
    y = np.mean([L4.run(dG, nuc_pos=L4.POS[idx[1]], seed=s) for s in (1, 2, 3)])
    y3.append(y); print(f'dG_PS = {g:+6.1f} kcal/mol   yield {y*100:5.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.plot(gs,  np.array(y12)*100, 'o-', lw=2, color='#0b6e5a', label='12 pac-sites (viral genome)')
ax.plot(gs3, np.array(y3)*100,  's-', lw=2, color='#b3402f', label='3 pac-sites (BruSafe cargo)')
ax.set_xlabel(r'pac-site affinity $\Delta G_{PS}$ (kcal/mol)')
ax.set_ylabel('packaged yield (%)')
ax.set_title('The optimum reverses with signal density')
ax.legend(frameon=False, fontsize=9); ax.invert_xaxis()
fig.tight_layout(); fig.savefig('M4_L4_affinity_scan.png', dpi=200)

## 4. Signal count as a design variable

In this model, adding pac-sites to the cargo increases yield more than tuning the affinity of the existing ones. The cell below quantifies the trade-off. The result is directly actionable, because adding a pac-site is a cloning step.

In [ ]:
g_fix = -6.0
counts = [1, 3, 6, 12]
yc = []
for n in counts:
    sites = list(np.linspace(0, 11, n).astype(int))
    dG = [-0.05]*12
    for i in sites: dG[i] = g_fix
    y = np.mean([L4.run(dG, nuc_pos=L4.POS[sites[len(sites)//2]], seed=s) for s in (1, 2, 3)])
    yc.append(y); print(f'{n:2d} pac-sites   yield {y*100:5.1f}%')

fig, ax = plt.subplots(figsize=(5.4, 3.7))
ax.plot(counts, np.array(yc)*100, 'o-', lw=2, color='#0b6e5a')
ax.set_xlabel('number of pac-sites on the cargo'); ax.set_ylabel('packaged yield (%)')
ax.set_title(fr'Yield vs signal count at $\Delta G_{{PS}}$ = {g_fix} kcal/mol')
fig.tight_layout(); fig.savefig('M4_L4_site_count.png', dpi=200)

---
**Outputs:** `M4_L4_affinity_scan.png`, `M4_L4_site_count.png`.

**Caveat.** The source paper does not state its concentration-conversion factor, so the absolute yields are underdetermined. The trends and reversals are the informative result; the absolute percentages should not be interpreted, nor tuned to match a measurement.